# Cross-Platform Analytics

Compare social media platforms side by side: posts, languages, hashtags, and GDELT events.

**What this notebook covers:**
- Platform summary and post volume comparison
- Language distribution across platforms
- GDELT vs social media timeline comparison
- Hashtag analysis across platforms
- LLM-assisted platform comparison

In [ ]:
from workspace_utils import *
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

sns.set_theme(style='whitegrid')
print('Libraries loaded.')

## 1. Platform Summary

In [ ]:
# Load platform summary from the database
summary = platform_summary()
print(f'Platforms in database: {len(summary)}')
summary

In [ ]:
# Side-by-side bar chart: posts per platform
post_col = [c for c in summary.columns if 'post' in c.lower() or 'count' in c.lower()][0]
platform_col = [c for c in summary.columns if 'platform' in c.lower() or 'media' in c.lower()][0]

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('husl', len(summary))
bars = ax.bar(summary[platform_col], summary[post_col], color=colors)

ax.set_title('Posts per Platform', fontsize=14)
ax.set_xlabel('Platform')
ax.set_ylabel('Number of Posts')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

# Add count labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
            f'{int(height):,}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 2. Language Distribution Across Platforms

In [ ]:
# Language breakdown by platform
lang_df = query_df("""
    SELECT p.media_type, ld.language, COUNT(*) as count
    FROM silver.posts p
    LEFT JOIN language_detection ld ON p.post_id = ld.document_id
    WHERE ld.language IS NOT NULL
    GROUP BY p.media_type, ld.language
    ORDER BY count DESC
    LIMIT 20
""")

print(f'Language-platform combinations: {len(lang_df)}')
lang_df.head(10)

In [ ]:
# Stacked bar chart: languages per platform
pivot = lang_df.pivot_table(
    index='media_type', columns='language', values='count', fill_value=0
)

# Keep only top languages for readability
top_langs = lang_df.groupby('language')['count'].sum().nlargest(8).index.tolist()
pivot_filtered = pivot[pivot.columns.intersection(top_langs)]

fig, ax = plt.subplots(figsize=(12, 6))
pivot_filtered.plot(
    kind='bar', stacked=True, ax=ax,
    colormap='tab20'
)
ax.set_title('Language Distribution by Platform', fontsize=14)
ax.set_xlabel('Platform')
ax.set_ylabel('Number of Posts')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.legend(title='Language', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 3. GDELT vs Social Media Timeline

In [ ]:
# GDELT events over time
gdelt_timeline = query_df("""
    SELECT DATE(dateadded) as event_date, COUNT(*) as event_count
    FROM gdelt_events
    WHERE dateadded >= NOW() - INTERVAL '90 days'
    GROUP BY DATE(dateadded)
    ORDER BY event_date
""")

# Social posts over time
social_timeline = query_df("""
    SELECT DATE(created_at) as post_date, COUNT(*) as post_count
    FROM silver.posts
    WHERE created_at >= NOW() - INTERVAL '90 days'
    GROUP BY DATE(created_at)
    ORDER BY post_date
""")

# Plot both on the same timeline
fig, ax1 = plt.subplots(figsize=(14, 6))

if len(gdelt_timeline) > 0:
    ax1.plot(pd.to_datetime(gdelt_timeline['event_date']),
             gdelt_timeline['event_count'],
             color='steelblue', linewidth=1.5, label='GDELT Events')
ax1.set_xlabel('Date')
ax1.set_ylabel('GDELT Events', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

ax2 = ax1.twinx()
if len(social_timeline) > 0:
    ax2.plot(pd.to_datetime(social_timeline['post_date']),
             social_timeline['post_count'],
             color='coral', linewidth=1.5, label='Social Posts')
ax2.set_ylabel('Social Posts', color='coral')
ax2.tick_params(axis='y', labelcolor='coral')

ax1.set_title('GDELT Events vs Social Media Posts (Last 90 Days)', fontsize=14)
fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.92))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 4. Hashtag Analysis

In [ ]:
# Top hashtags across all platforms
hashtags = top_hashtags()
print(f'Top hashtags found: {len(hashtags)}')

# Bar chart of top hashtags
top_ht = hashtags.head(20)
count_col = [c for c in top_ht.columns if 'count' in c.lower()][0]
tag_col = [c for c in top_ht.columns if 'hashtag' in c.lower() or 'tag' in c.lower()][0]

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(top_ht[tag_col].astype(str), top_ht[count_col],
        color=sns.color_palette('rocket', len(top_ht)))
ax.set_xlabel('Usage Count')
ax.set_title('Top 20 Hashtags', fontsize=14)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. LLM Platform Comparison

In [ ]:
# Ask the LLM to compare platforms based on the summary data
analysis = analyze_dataframe(
    summary,
    "Compare the social media platforms by engagement and content themes. "
    "Which platform is most active? What are the key differences between them? "
    "What strategies might explain the differences in post volume and engagement?"
)
print(analysis)

In [ ]:
# Summary
print('Cross-Platform Analytics Complete!')
print(f'  Platforms analyzed: {len(summary)}')
if len(gdelt_timeline) > 0:
    print(f'  GDELT events (90 days): {gdelt_timeline["event_count"].sum():,}')
if len(social_timeline) > 0:
    print(f'  Social posts (90 days): {social_timeline["post_count"].sum():,}')
print(f'  Top hashtags: {len(hashtags)}')
print('\nNext: try topic_modeling.ipynb or image_analysis.ipynb')